# Movie Recommendation System — Phase 1: Data Loading & EDA
**Dataset:** CiaoDVD (`movie-ratings.txt`)  
**Course:** CSIT 360 — Advanced Techniques in Data Science  
**Spring 2026**

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Create output folders if they don't exist
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Setup complete.')

## 2. Load the Dataset

The `movie-ratings.txt` file is **tab-separated** with columns:  
`userId, movieId, movie-categoryId, reviewId, movieRating, reviewDate`

In [ ]:
# --- EDIT THIS PATH if your file is somewhere else ---
DATA_PATH = '../data/raw/movie-ratings.txt'

col_names = ['userId', 'movieId', 'categoryId', 'reviewId', 'rating', 'reviewDate']

df = pd.read_csv(
    DATA_PATH,
    sep='\t',          # tab-separated
    header=None,
    names=col_names
)

print(f'Loaded {len(df):,} rows.')
df.head(10)

## 3. Basic Statistics

In [ ]:
n_users   = df['userId'].nunique()
n_movies  = df['movieId'].nunique()
n_ratings = len(df)

# Sparsity: fraction of the user-movie matrix that is EMPTY
total_possible = n_users * n_movies
sparsity = 1 - (n_ratings / total_possible)

print('=' * 40)
print(f'  Number of users   : {n_users:,}')
print(f'  Number of movies  : {n_movies:,}')
print(f'  Number of ratings : {n_ratings:,}')
print(f'  Matrix size       : {n_users:,} x {n_movies:,} = {total_possible:,}')
print(f'  Sparsity          : {sparsity:.4%}')
print('=' * 40)

## 4. Data Quality Check

In [ ]:
print('--- Missing values ---')
print(df.isnull().sum())

print(f'\n--- Duplicate rows ---')
n_dups = df.duplicated().sum()
print(f'  Duplicates found: {n_dups}')

print(f'\n--- Rating range ---')
print(f'  Min: {df["rating"].min()}  |  Max: {df["rating"].max()}')

# Drop duplicates if any
if n_dups > 0:
    df = df.drop_duplicates()
    print(f'  Dropped {n_dups} duplicate rows. Remaining: {len(df):,}')
else:
    print('  No duplicates — data is clean!')

## 5. Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

rating_counts = df['rating'].value_counts().sort_index()
bars = ax.bar(
    rating_counts.index,
    rating_counts.values,
    color=sns.color_palette('muted', 5),
    edgecolor='white',
    width=0.6
)

# Annotate each bar with count and percentage
total = rating_counts.sum()
for bar, count in zip(bars, rating_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f'{count:,}\n({count/total:.1%})',
        ha='center', va='bottom', fontsize=9
    )

ax.set_xlabel('Star Rating', fontsize=12)
ax.set_ylabel('Number of Ratings', fontsize=12)
ax.set_title('Distribution of Ratings (CiaoDVD)', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../results/figures/rating_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → results/figures/rating_distribution.png')

## 6. Long-Tail Distributions
### 6a. Ratings per User

In [ ]:
ratings_per_user = df.groupby('userId')['rating'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: full long-tail
axes[0].bar(range(len(ratings_per_user)), ratings_per_user.values, color='steelblue', width=1)
axes[0].set_xlabel('Users (sorted by activity)', fontsize=11)
axes[0].set_ylabel('Number of Ratings', fontsize=11)
axes[0].set_title('Long-Tail: Ratings per User', fontsize=13, fontweight='bold')

# Right: histogram
axes[1].hist(ratings_per_user.values, bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Ratings per User', fontsize=11)
axes[1].set_ylabel('Number of Users', fontsize=11)
axes[1].set_title('Distribution of User Activity', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/ratings_per_user.png', bbox_inches='tight')
plt.show()

print(f'Median ratings per user : {ratings_per_user.median():.0f}')
print(f'Mean   ratings per user : {ratings_per_user.mean():.1f}')
print(f'Max    ratings per user : {ratings_per_user.max()}')

### 6b. Ratings per Movie

In [ ]:
ratings_per_movie = df.groupby('movieId')['rating'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(len(ratings_per_movie)), ratings_per_movie.values, color='coral', width=1)
axes[0].set_xlabel('Movies (sorted by popularity)', fontsize=11)
axes[0].set_ylabel('Number of Ratings', fontsize=11)
axes[0].set_title('Long-Tail: Ratings per Movie', fontsize=13, fontweight='bold')

axes[1].hist(ratings_per_movie.values, bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Ratings per Movie', fontsize=11)
axes[1].set_ylabel('Number of Movies', fontsize=11)
axes[1].set_title('Distribution of Movie Popularity', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/ratings_per_movie.png', bbox_inches='tight')
plt.show()

print(f'Median ratings per movie : {ratings_per_movie.median():.0f}')
print(f'Mean   ratings per movie : {ratings_per_movie.mean():.1f}')
print(f'Max    ratings per movie : {ratings_per_movie.max()}')

## 7. Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Total Ratings', 'Unique Users', 'Unique Movies',
        'Sparsity', 'Avg Rating', 'Median Ratings / User', 'Median Ratings / Movie'
    ],
    'Value': [
        f'{n_ratings:,}',
        f'{n_users:,}',
        f'{n_movies:,}',
        f'{sparsity:.4%}',
        f'{df["rating"].mean():.3f}',
        f'{ratings_per_user.median():.0f}',
        f'{ratings_per_movie.median():.0f}'
    ]
})

summary

## 8. Save Cleaned Data

In [ ]:
# Keep only the columns needed for modeling
df_clean = df[['userId', 'movieId', 'rating']].copy()
df_clean.to_csv('../data/processed/ratings_clean.csv', index=False)

print(f'Saved cleaned data: {len(df_clean):,} rows → data/processed/ratings_clean.csv')
df_clean.head()

## EDA Complete ✅

**Key Findings:**
- The dataset is highly **sparse** (most users have rated very few movies)
- Ratings skew toward **higher values** (4s and 5s most common)
- Strong **long-tail** effect: a few popular movies/active users dominate
- No significant data quality issues found

Next step → `02_cf_model.ipynb` for recommendation algorithms.